In [31]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum, avg, min, max, desc, asc, lit, round

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Clase3-Spark-DataFrames-SQL")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")


In [32]:
emp = [
    (1, "AAA", "dept1", 1000),
    (2, "BBB", "dept1", 1100),
    (3, "CCC", "dept1", 3000),
    (4, "DDD", "dept1", 1500),
    (5, "EEE", "dept2", 8000),
    (6, "FFF", "dept2", 7200),
    (7, "GGG", "dept3", 7100),
    (8, "HHH", "dept3", 3700),
    (9, "III", "dept3", 4500),
    (10, "JJJ", "dept5", 3400),
]

dept = [
    ("dept1", "Department - 1"),
    ("dept2", "Department - 2"),
    ("dept3", "Department - 3"),
    ("dept4", "Department - 4"),
]

df = spark.createDataFrame(emp, ["id", "name", "dept", "salary"])
deptdf = spark.createDataFrame(dept, ["id", "name"])

df.show()
deptdf.show()


+---+----+-----+------+
| id|name| dept|salary|
+---+----+-----+------+
|  1| AAA|dept1|  1000|
|  2| BBB|dept1|  1100|
|  3| CCC|dept1|  3000|
|  4| DDD|dept1|  1500|
|  5| EEE|dept2|  8000|
|  6| FFF|dept2|  7200|
|  7| GGG|dept3|  7100|
|  8| HHH|dept3|  3700|
|  9| III|dept3|  4500|
| 10| JJJ|dept5|  3400|
+---+----+-----+------+

+-----+--------------+
|   id|          name|
+-----+--------------+
|dept1|Department - 1|
|dept2|Department - 2|
|dept3|Department - 3|
|dept4|Department - 4|
+-----+--------------+



3) Operaciones básicas: count / columns / dtypes / schema / printSchema

In [33]:
df.count()

10

In [34]:
df.columns

['id', 'name', 'dept', 'salary']

In [35]:
df.dtypes


[('id', 'bigint'),
 ('name', 'string'),
 ('dept', 'string'),
 ('salary', 'bigint')]

In [36]:
df.schema


StructType([StructField('id', LongType(), True), StructField('name', StringType(), True), StructField('dept', StringType(), True), StructField('salary', LongType(), True)])

In [37]:
df.printSchema()


root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- salary: long (nullable = true)



4) select

In [38]:
df.select("name", "salary").show()

+----+------+
|name|salary|
+----+------+
| AAA|  1000|
| BBB|  1100|
| CCC|  3000|
| DDD|  1500|
| EEE|  8000|
| FFF|  7200|
| GGG|  7100|
| HHH|  3700|
| III|  4500|
| JJJ|  3400|
+----+------+



In [39]:
df.select(col("dept").alias("department"), col("salary")).show()

+----------+------+
|department|salary|
+----------+------+
|     dept1|  1000|
|     dept1|  1100|
|     dept1|  3000|
|     dept1|  1500|
|     dept2|  8000|
|     dept2|  7200|
|     dept3|  7100|
|     dept3|  3700|
|     dept3|  4500|
|     dept5|  3400|
+----------+------+



In [40]:
df.select("*").show(5)

+---+----+-----+------+
| id|name| dept|salary|
+---+----+-----+------+
|  1| AAA|dept1|  1000|
|  2| BBB|dept1|  1100|
|  3| CCC|dept1|  3000|
|  4| DDD|dept1|  1500|
|  5| EEE|dept2|  8000|
+---+----+-----+------+
only showing top 5 rows


5) filter + isin (esto aparece en tu notebook)

In [41]:
df.filter(col("salary") >= 3000).show()

+---+----+-----+------+
| id|name| dept|salary|
+---+----+-----+------+
|  3| CCC|dept1|  3000|
|  5| EEE|dept2|  8000|
|  6| FFF|dept2|  7200|
|  7| GGG|dept3|  7100|
|  8| HHH|dept3|  3700|
|  9| III|dept3|  4500|
| 10| JJJ|dept5|  3400|
+---+----+-----+------+



In [42]:
lista = [1, 2, 3]
df.filter(col("id").isin(lista)).show()


+---+----+-----+------+
| id|name| dept|salary|
+---+----+-----+------+
|  1| AAA|dept1|  1000|
|  2| BBB|dept1|  1100|
|  3| CCC|dept1|  3000|
+---+----+-----+------+



6) drop

In [43]:
df.drop("salary").show()

+---+----+-----+
| id|name| dept|
+---+----+-----+
|  1| AAA|dept1|
|  2| BBB|dept1|
|  3| CCC|dept1|
|  4| DDD|dept1|
|  5| EEE|dept2|
|  6| FFF|dept2|
|  7| GGG|dept3|
|  8| HHH|dept3|
|  9| III|dept3|
| 10| JJJ|dept5|
+---+----+-----+



7) Aggregations (groupBy + agg)

In [44]:
df.groupBy("dept").agg(
    count("salary").alias("count"),
    sum("salary").alias("sum"),
    max("salary").alias("max"),
    min("salary").alias("min"),
    avg("salary").alias("avg")
).show()


+-----+-----+-----+----+----+------+
| dept|count|  sum| max| min|   avg|
+-----+-----+-----+----+----+------+
|dept1|    4| 6600|3000|1000|1650.0|
|dept2|    2|15200|8000|7200|7600.0|
|dept3|    3|15300|7100|3700|5100.0|
|dept5|    1| 3400|3400|3400|3400.0|
+-----+-----+-----+----+----+------+



8) Sorting

In [45]:
df.sort("salary").show()

+---+----+-----+------+
| id|name| dept|salary|
+---+----+-----+------+
|  1| AAA|dept1|  1000|
|  2| BBB|dept1|  1100|
|  4| DDD|dept1|  1500|
|  3| CCC|dept1|  3000|
| 10| JJJ|dept5|  3400|
|  8| HHH|dept3|  3700|
|  9| III|dept3|  4500|
|  7| GGG|dept3|  7100|
|  6| FFF|dept2|  7200|
|  5| EEE|dept2|  8000|
+---+----+-----+------+



In [46]:
df.sort(desc("salary")).show()

+---+----+-----+------+
| id|name| dept|salary|
+---+----+-----+------+
|  5| EEE|dept2|  8000|
|  6| FFF|dept2|  7200|
|  7| GGG|dept3|  7100|
|  9| III|dept3|  4500|
|  8| HHH|dept3|  3700|
| 10| JJJ|dept5|  3400|
|  3| CCC|dept1|  3000|
|  4| DDD|dept1|  1500|
|  2| BBB|dept1|  1100|
|  1| AAA|dept1|  1000|
+---+----+-----+------+



In [47]:
df.orderBy(desc("salary")).show()

+---+----+-----+------+
| id|name| dept|salary|
+---+----+-----+------+
|  5| EEE|dept2|  8000|
|  6| FFF|dept2|  7200|
|  7| GGG|dept3|  7100|
|  9| III|dept3|  4500|
|  8| HHH|dept3|  3700|
| 10| JJJ|dept5|  3400|
|  3| CCC|dept1|  3000|
|  4| DDD|dept1|  1500|
|  2| BBB|dept1|  1100|
|  1| AAA|dept1|  1000|
+---+----+-----+------+



In [48]:
df.orderBy(asc("salary")).show()


+---+----+-----+------+
| id|name| dept|salary|
+---+----+-----+------+
|  1| AAA|dept1|  1000|
|  2| BBB|dept1|  1100|
|  4| DDD|dept1|  1500|
|  3| CCC|dept1|  3000|
| 10| JJJ|dept5|  3400|
|  8| HHH|dept3|  3700|
|  9| III|dept3|  4500|
|  7| GGG|dept3|  7100|
|  6| FFF|dept2|  7200|
|  5| EEE|dept2|  8000|
+---+----+-----+------+



9) Columnas derivadas (bonus)

In [49]:
df_bonus = df.withColumn("bonus_10pct", round(col("salary") * 0.10, 2))
df_bonus.show()

+---+----+-----+------+-----------+
| id|name| dept|salary|bonus_10pct|
+---+----+-----+------+-----------+
|  1| AAA|dept1|  1000|      100.0|
|  2| BBB|dept1|  1100|      110.0|
|  3| CCC|dept1|  3000|      300.0|
|  4| DDD|dept1|  1500|      150.0|
|  5| EEE|dept2|  8000|      800.0|
|  6| FFF|dept2|  7200|      720.0|
|  7| GGG|dept3|  7100|      710.0|
|  8| HHH|dept3|  3700|      370.0|
|  9| III|dept3|  4500|      450.0|
| 10| JJJ|dept5|  3400|      340.0|
+---+----+-----+------+-----------+



10) Joins (inner / left / right / full) — versión limpia

Inner Join

In [50]:
inner_join = (
    df.alias("a")
    .join(deptdf.alias("b"), col("a.dept") == col("b.id"), "inner")
    .select(
        col("a.id").alias("emp_id"),
        col("a.name").alias("emp_name"),
        col("a.dept").alias("dept_id"),
        col("b.name").alias("dept_name"),
        col("a.salary")
    )
)
inner_join.show()


+------+--------+-------+--------------+------+
|emp_id|emp_name|dept_id|     dept_name|salary|
+------+--------+-------+--------------+------+
|     1|     AAA|  dept1|Department - 1|  1000|
|     2|     BBB|  dept1|Department - 1|  1100|
|     3|     CCC|  dept1|Department - 1|  3000|
|     4|     DDD|  dept1|Department - 1|  1500|
|     5|     EEE|  dept2|Department - 2|  8000|
|     6|     FFF|  dept2|Department - 2|  7200|
|     7|     GGG|  dept3|Department - 3|  7100|
|     8|     HHH|  dept3|Department - 3|  3700|
|     9|     III|  dept3|Department - 3|  4500|
+------+--------+-------+--------------+------+



Left Outer Join

In [51]:
df.join(deptdf, df["dept"] == deptdf["id"], "left").show()

+---+----+-----+------+-----+--------------+
| id|name| dept|salary|   id|          name|
+---+----+-----+------+-----+--------------+
|  1| AAA|dept1|  1000|dept1|Department - 1|
|  2| BBB|dept1|  1100|dept1|Department - 1|
|  3| CCC|dept1|  3000|dept1|Department - 1|
|  4| DDD|dept1|  1500|dept1|Department - 1|
|  5| EEE|dept2|  8000|dept2|Department - 2|
|  6| FFF|dept2|  7200|dept2|Department - 2|
|  7| GGG|dept3|  7100|dept3|Department - 3|
|  8| HHH|dept3|  3700|dept3|Department - 3|
| 10| JJJ|dept5|  3400| NULL|          NULL|
|  9| III|dept3|  4500|dept3|Department - 3|
+---+----+-----+------+-----+--------------+



Right Outer Join

In [52]:
df.join(deptdf, df["dept"] == deptdf["id"], "right").show()

+----+----+-----+------+-----+--------------+
|  id|name| dept|salary|   id|          name|
+----+----+-----+------+-----+--------------+
|   4| DDD|dept1|  1500|dept1|Department - 1|
|   3| CCC|dept1|  3000|dept1|Department - 1|
|   2| BBB|dept1|  1100|dept1|Department - 1|
|   1| AAA|dept1|  1000|dept1|Department - 1|
|   6| FFF|dept2|  7200|dept2|Department - 2|
|   5| EEE|dept2|  8000|dept2|Department - 2|
|   9| III|dept3|  4500|dept3|Department - 3|
|   8| HHH|dept3|  3700|dept3|Department - 3|
|   7| GGG|dept3|  7100|dept3|Department - 3|
|NULL|NULL| NULL|  NULL|dept4|Department - 4|
+----+----+-----+------+-----+--------------+



Full Outer Join

In [53]:
df.join(deptdf, df["dept"] == deptdf["id"], "full").show()

+----+----+-----+------+-----+--------------+
|  id|name| dept|salary|   id|          name|
+----+----+-----+------+-----+--------------+
|   1| AAA|dept1|  1000|dept1|Department - 1|
|   2| BBB|dept1|  1100|dept1|Department - 1|
|   3| CCC|dept1|  3000|dept1|Department - 1|
|   4| DDD|dept1|  1500|dept1|Department - 1|
|   5| EEE|dept2|  8000|dept2|Department - 2|
|   6| FFF|dept2|  7200|dept2|Department - 2|
|   7| GGG|dept3|  7100|dept3|Department - 3|
|   8| HHH|dept3|  3700|dept3|Department - 3|
|   9| III|dept3|  4500|dept3|Department - 3|
|NULL|NULL| NULL|  NULL|dept4|Department - 4|
|  10| JJJ|dept5|  3400| NULL|          NULL|
+----+----+-----+------+-----+--------------+



11) Spark SQL (Temp View) — reemplazando lo Databricks/Hive

In [54]:
df.createOrReplaceTempView("empleados")

In [55]:
spark.sql("SELECT DISTINCT dept FROM empleados").show()

+-----+
| dept|
+-----+
|dept1|
|dept2|
|dept3|
|dept5|
+-----+



In [56]:
spark.sql("SELECT * FROM empleados WHERE salary >= 1500 ORDER BY salary DESC").show()

+---+----+-----+------+
| id|name| dept|salary|
+---+----+-----+------+
|  5| EEE|dept2|  8000|
|  6| FFF|dept2|  7200|
|  7| GGG|dept3|  7100|
|  9| III|dept3|  4500|
|  8| HHH|dept3|  3700|
| 10| JJJ|dept5|  3400|
|  3| CCC|dept1|  3000|
|  4| DDD|dept1|  1500|
+---+----+-----+------+



12) CSV (ejercicio equivalente, pero local)

In [58]:
import pandas as pd
from pathlib import Path

out_dir = Path("data")
out_dir.mkdir(exist_ok=True)

csv_path = out_dir / "paciente.csv"

pd.DataFrame({
    "id": [1, 2, 3],
    "nombre": ["Ana", "Luis", "Sofia"],
    "edad": [23, 31, 27]
}).to_csv(csv_path, index=False)

df_paciente = spark.read.csv(str(csv_path), header=True, inferSchema=True)
df_paciente.show()


+---+------+----+
| id|nombre|edad|
+---+------+----+
|  1|   Ana|  23|
|  2|  Luis|  31|
|  3| Sofia|  27|
+---+------+----+



In [59]:
df_paciente.write.mode("overwrite").option("header", True).csv("data/paciente_out_csv")

In [60]:
spark.stop()